# knot — 02: ingest

Bind a source to a class. See the write SQL knot emits. Bind row
data to it via the connector. Verify what landed.

In [2]:
import uuid

import pandas as pd
import psycopg

In [3]:
# Same shared spec as 01_deploy — imports the spec, the Movie class,
# the imdb Source, and the imdb→Movie binding from ``movies_spec.py``.
# Real deployments share this exact import pattern: workers, the
# service API, the ER pipeline all import from the same spec module.
from movies_spec import imdb, imdb_movie_b, movie, spec

imdb_movie_b

SourceBinding(source=Source(name='imdb', description=None), class_=OntologyClass(name='Movie', kind=<ClassKind.CONCRETE: 'concrete'>, is_a=None, mixins=[], slots=[Slot(name='canonical_id', type=<Primitive.TEXT: 'text'>, identifier=True, required=True, description=None), Slot(name='title', type=<Primitive.TEXT: 'text'>, identifier=False, required=True, description=None), Slot(name='year', type=<Primitive.INTEGER: 'integer'>, identifier=False, required=False, description=None), Slot(name='director', type=ClassRef(target=OntologyClass(name='Person', kind=<ClassKind.CONCRETE: 'concrete'>, is_a=None, mixins=[], slots=[Slot(name='canonical_id', type=<Primitive.TEXT: 'text'>, identifier=True, required=True, description=None), Slot(name='name', type=<Primitive.TEXT: 'text'>, identifier=False, required=True, description=None), Slot(name='birth_country', type=<Primitive.TEXT: 'text'>, identifier=False, required=False, description=None)], description=None)), identifier=False, required=False, desc

In [4]:
# Host plumbing + deploy. Schema name is a throwaway per-run id.
pg = psycopg.connect(
    host="localhost",
    port=5433,
    user="knot",
    password="knot",
    dbname="knot",
    autocommit=True,
)
schema = f"knot_play_{uuid.uuid4().hex[:8]}"
pg.execute(spec.ddl(schema=schema))
schema

'knot_play_f598983a'

In [5]:
# Load real sample data from disk — imdb's movies.json. Each row
# carries:
#   * `source_identifier` — imdb's own key (e.g. "tt1838941").
#     This is the only stable identity imdb knows about; knot's
#     cross-source `canonical_id` doesn't exist yet — ER assigns
#     it later (see 04_er).
#   * the class slots (`title`, `year`, `director`) as native values.
#   * extras (`imdb_rating`, `num_votes`, `box_office_usd`, …) that
#     aren't in the spec — they ride along in the row dict and land
#     in `raw_payload jsonb` on the binding row, recoverable later
#     without re-fetching from imdb.
raw_df = pd.read_json("../data/movies/imdb/movies.json")
print(f"loaded {len(raw_df)} rows")
raw_df.head()

loaded 70 rows


,source_identifier,title,year,runtime_minutes,director,imdb_rating,num_votes,mpaa_rating,aspect_ratio,color_info,box_office_usd,country_of_origin
0,tt1838941,Reservoir Dogs,1992,98,p_tarantino,7.8,1651568,G,1.43 : 1 (IMAX),Color,1.384664e+09,[USA]
1,tt2878306,Kill Bill: Vol. 1,2003,112,p_tarantino,9.5,1757925,Not Rated,1.66 : 1,Color,3.652510e+08,[USA]
2,tt5322801,Inglourious Basterds,2009,152,p_tarantino,6.8,167708,NC-17,1.85 : 1,Color,1.057024e+09,[USA]
3,tt0768738,Django Unchained,2012,166,p_tarantino,8.4,1294369,PG,2.39 : 1,Color,1.386899e+09,[USA]
4,tt9462920,Seven Samurai,1954,207,p_kurosawa,9.4,27589,Not Rated,1.37 : 1,Black and White,NaN,[USA]


In [6]:
# `binding.write_sql()` returns two SQL templates — both reference a
# single `%(rows)s::jsonb` parameter. knot never touches the rows;
# the host's connector binds them at execute time.
close_out, insert = imdb_movie_b.write_sql(schema=schema)
print("--- close_out ---")
print(close_out)
print("\n--- insert ---")
print(insert)

--- close_out ---
UPDATE knot_play_f598983a.movie_bindings AS b
SET valid_to = now()
FROM (
    SELECT
        (r->>'canonical_id') AS canonical_id,
        (r->>'source_identifier') AS source_identifier
    FROM jsonb_array_elements(%(rows)s::jsonb) AS r
) AS keys
WHERE b.canonical_id = keys.canonical_id
  AND b.source_name = 'imdb'
  AND b.source_identifier = keys.source_identifier
  AND b.valid_to IS NULL;

--- insert ---
INSERT INTO knot_play_f598983a.movie_bindings (source_name, source_identifier, canonical_id, title, year, director, title_embedding, raw_payload)
SELECT
    'imdb',
    raw.source_identifier,
    raw.canonical_id::text,
    raw.title::text,
    raw.year::integer,
    raw.director::text,
    raw.title_embedding::vector(384),
    raw.__raw_payload
FROM (
    SELECT
        r AS __raw_payload,
        (r->>'source_identifier') AS source_identifier,
        (r->>'canonical_id') AS canonical_id,
        (r->>'title') AS title,
        (r->>'year') AS year,
        (r->>

In [7]:
# Run both statements with the rows bound as a single jsonb param.
# raw_df → JSON via DataFrame.to_json (records orientation = a JSON
# array of dicts, which is what jsonb_array_elements expects).
# For an autocommit connection each cur.execute commits independently;
# wrap in pg.transaction() if you want atomic close_out + insert.
payload = raw_df.to_json(orient="records")
with pg.cursor() as _cur:
    _cur.execute(close_out, {"rows": payload})
    _cur.execute(insert, {"rows": payload})

In [8]:
# Verify via ``movie.from_source(imdb)`` — one source's claims about
# Movie. This is a Query over the raw bindings layer (one row per
# source_identifier) scoped to ``source_name = 'imdb'``. The
# ``resolved`` layer would be empty here: ER hasn't run yet, so
# every row's ``canonical_id`` is still NULL.
q = (
    movie.from_source(imdb)
    .order_by(movie.col.year, "desc")
    .limit(10)
    .select(movie.col.canonical_id, movie.col.title, movie.col.year)
)
pd.read_sql_query(q.sql(schema=schema), pg)

/tmp/ipykernel_2732971/2659400726.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql_query(q.sql(schema=schema), pg)


,canonical_id,title,year
0,None,Dune: Part Two,2024
1,None,Oppenheimer,2023
2,None,Poor Things,2023
3,None,Barbie,2023
4,None,The Northman,2022
5,None,Dune,2021
6,None,Midsommar,2019
7,None,Little Women,2019
8,None,The Lighthouse,2019
9,None,Pain and Glory,2019
